In [0]:
dbutils.widgets.text("dataset_path", "", "Dataset path (csv/json/parquet)")
dbutils.widgets.dropdown("file_type", "csv", ["csv", "json", "parquet"], "File type")

dbutils.widgets.dropdown("run_stats", "yes", ["yes", "no"], "Run Descriptive Stats?")
dbutils.widgets.dropdown("run_ml", "yes", ["yes", "no"], "Run ML Jobs?")
dbutils.widgets.dropdown("run_benchmark", "yes", ["yes", "no"], "Run Benchmark (1/2/4/8)?")

dbutils.widgets.text("kmeans_k", "5", "KMeans K")
dbutils.widgets.text("ml_sample_fraction", "0.20", "ML sample fraction (0.1-1.0)")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, TimestampType, DateType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.clustering import KMeans
import time, json, re

# --------------------------
# 1) Read widget parameters
# --------------------------
dataset_path = dbutils.widgets.get("dataset_path").strip()
file_type = dbutils.widgets.get("file_type").strip().lower()
run_stats = dbutils.widgets.get("run_stats").strip().lower() == "yes"
run_ml = dbutils.widgets.get("run_ml").strip().lower() == "yes"
run_benchmark = dbutils.widgets.get("run_benchmark").strip().lower() == "yes"
kmeans_k = int(dbutils.widgets.get("kmeans_k"))
ml_frac = float(dbutils.widgets.get("ml_sample_fraction"))

if not dataset_path:
    raise ValueError("Please provide dataset_path in the widget.")

BASE_DIR = "/Volumes/workspace/default/datasets"

# try to guess file type from extension
path_lower = dataset_path.lower()
if path_lower.endswith(".parquet"):
    file_type = "parquet"
elif path_lower.endswith(".json"):
    file_type = "json"
elif path_lower.endswith(".csv") or path_lower.endswith(".csv.gz"):
    file_type = "csv"

print(f"Using dataset: {dataset_path} (type = {file_type})")

# --------------------------
# 2) Load and validate data
# --------------------------
if file_type == "csv":
    df = (
        spark.read
        .option("header", False)
        .option("inferSchema", True)
        .csv(dataset_path)
    )
elif file_type == "json":
    df = spark.read.option("inferSchema", True).json(dataset_path)
elif file_type == "parquet":
    df = spark.read.parquet(dataset_path)
else:
    raise ValueError("Unsupported file_type. Use csv / json / parquet.")

print("\n---------------------------------------------- Dataset Validation ----------------------------------------------")

# empty file
if len(df.take(1)) == 0:
    raise ValueError("Dataset is empty. Please upload a file with rows.")

# very small number of columns
col_count = len(df.columns)
if col_count < 2:
    raise ValueError(f"Dataset has only {col_count} column(s). Need more columns.")

# check numeric columns (for ML part)
numeric_cols_check = [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]
if run_ml and len(numeric_cols_check) < 2:
    raise ValueError("Need at least 2 numeric columns to run ML jobs.")

print("✔️ Basic validation passed.")

if run_stats:
    row_count = df.count()
    MIN_ROWS = 100
    if row_count < MIN_ROWS:
        raise ValueError(f"Dataset has {row_count} rows (< {MIN_ROWS}). Please use a larger file.")
    print(f"Rows: {row_count:,}   |   Columns: {col_count}   |   Numeric columns: {len(numeric_cols_check)}")

# --------------------------
# 3) Clean column names
# --------------------------
def clean_column_name(col):
    col = col.lower()
    col = re.sub(r"[^\w]+", "_", col)
    col = re.sub(r"_+", "_", col)
    return col.strip("_")

orig_cols = df.columns
new_cols = [clean_column_name(c) for c in orig_cols]
df = df.toDF(*new_cols)

# --------------------------
# 4) Detect numeric / time columns
# --------------------------
numeric_cols = []
timestamp_cols = []

for f in df.schema.fields:
    if isinstance(f.dataType, NumericType):
        numeric_cols.append(f.name)
    elif isinstance(f.dataType, (TimestampType, DateType)):
        timestamp_cols.append(f.name)

# cast timestamps to long so they can also be used as numeric features
for c in timestamp_cols:
    df = df.withColumn(c, F.col(c).cast("long"))

numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]

print("\n---------------------------------------------- Column Overview ----------------------------------------------")
print("Numeric columns:", numeric_cols[:20], "..." if len(numeric_cols) > 20 else "")
print("Total numeric columns:", len(numeric_cols))

if run_ml and len(numeric_cols) < 2:
    raise ValueError("Not enough numeric columns for ML after cleaning/casting.")

# to keep things simple we only use a few numeric columns
max_cols = 4
use_cols = numeric_cols[:max_cols]

# --------------------------
# 5) Prepare output folders
# --------------------------
job_id = f"job_{int(time.time())}"
out_base = f"{BASE_DIR}/results/{job_id}"
dbutils.fs.mkdirs(out_base)

run_base = f"{BASE_DIR}/runs/{job_id}"
input_dir = f"{run_base}/input"
dbutils.fs.mkdirs(input_dir)

print("\n---------------------------------------------- Paths ----------------------------------------------")
print("Results will be saved in:", out_base)
print("Materializing input will be saved in:", input_dir)

# --------------------------
# 6) Descriptive statistics
# --------------------------
if run_stats:
    print("\n---------------------------------------------- Descriptive Statistics ----------------------------------------------")

    total_rows = df.count()
    total_cols = len(df.columns)

    # (1) shape
    print("\n1️⃣ Data Shape")
    print("  • Total rows   :", total_rows)
    print("  • Total columns:", total_cols)

    # (2) data types
    print("\n2️⃣ Data types")
    dtype_rows = []
    for name, dtype in df.dtypes:
        if dtype in ["int", "bigint", "double", "float", "long", "decimal", "smallint", "tinyint"]:
            group = "numeric"
        elif dtype in ["string", "boolean", "date", "timestamp"]:
            group = "string"
        else:
            group = "other"
        dtype_rows.append((name, dtype, group))
    dtypes_df = spark.createDataFrame(dtype_rows, ["column", "spark_dtype", "group"])
    display(dtypes_df)

    # (3) summary for numeric columns
    print("\n3️⃣ Numeric summary")
    if numeric_cols:
        numeric_summary_df = df.select(*numeric_cols).summary("count", "mean", "stddev", "min", "max")
        display(numeric_summary_df)
        numeric_summary_df.write.mode("overwrite").parquet(f"{out_base}/numeric_summary.parquet")
    else:
        print("No numeric columns found.")

    # (4) null counts for first 50 columns
    print("\n4️⃣ Missing values (first 50 columns)")
    cols_for_nulls = df.columns[:50]
    dtypes_map = dict(df.dtypes)

    null_exprs = []
    for c in cols_for_nulls:
        dtype = dtypes_map[c]
        if dtype == "string":
            cond = F.col(c).isNull() | (F.trim(F.col(c)) == "")
        else:
            cond = F.col(c).isNull()
        null_exprs.append(F.sum(F.when(cond, 1).otherwise(0)).alias(c))

    null_counts = df.select(*null_exprs).collect()[0].asDict()
    null_rows = []
    for c in cols_for_nulls:
        cnt = int(null_counts.get(c, 0))
        pct = (cnt / total_rows * 100) if total_rows else 0.0
        null_rows.append((c, cnt, float(pct)))

    null_stats_df = spark.createDataFrame(null_rows, ["column", "null_count", "null_pct"])
    display(null_stats_df.orderBy(F.desc("null_pct")))

    # save stats to files
    dtypes_df.write.mode("overwrite").parquet(f"{out_base}/dtypes.parquet")
    null_stats_df.write.mode("overwrite").parquet(f"{out_base}/nulls.parquet")
    dbutils.fs.put(
        f"{out_base}/shape.json",
        json.dumps({"total_rows": total_rows, "total_columns": total_cols}, indent=2),
        overwrite=True,
    )

# --------------------------
# 7) ML jobs
# --------------------------
ml_results = {}

if run_ml:
    if len(use_cols) < 2:
        raise ValueError("Dataset needs at least 2 numeric columns for ML.")

    print("\n---------------------------------------------- ML Jobs ----------------------------------------------")

    df_num = df.select(*use_cols).dropna()
    df_num_path = f"{input_dir}/df_num.parquet"
    df_num.write.mode("overwrite").parquet(df_num_path)
    #print("Saved full numeric subset to:", df_num_path)

    df_small = spark.read.parquet(df_num_path).sample(False, ml_frac, seed=42)
    df_small_path = f"{input_dir}/df_small.parquet"
    df_small.write.mode("overwrite").parquet(df_small_path)
    #print("Saved sampled subset to:", df_small_path)

    target = use_cols[0]
    features = use_cols[1:]
    assembler = VectorAssembler(inputCols=features, outputCol="features", handleInvalid="skip")

    def get_features_df():
        """Read sampled data and add 'features' column."""
        s = spark.read.parquet(df_small_path)
        return assembler.transform(s).select(*use_cols, "features")

    #  Regression
    print("\n1️⃣ Regression")
    df_feat = get_features_df()
    df_reg = df_feat.select(F.col(target).alias("label"), "features")
    lr_model = LinearRegression(featuresCol="features", labelCol="label", maxIter=20).fit(df_reg)
    ml_results["regression"] = {
        "target": target,
        "features_used": features,
        "rmse": float(lr_model.summary.rootMeanSquaredError),
        "r2": float(lr_model.summary.r2),
    }
    print("  RMSE:", ml_results["regression"]["rmse"])
    print("  R2  :", ml_results["regression"]["r2"])

    # 7.4 Classification (median split)
    print("\n2️⃣ Classification")
    df_small_read = spark.read.parquet(df_small_path)
    median_val = df_small_read.approxQuantile(target, [0.5], 0.01)[0]

    df_feat = get_features_df()
    df_cls = (
        df_feat.withColumn("label", F.when(F.col(target) >= median_val, 1.0).otherwise(0.0))
        .select("label", "features")
    )
    distinct_labels = df_cls.select("label").distinct().count()

    if distinct_labels < 2:
        print("  Only one class found. Skipping classification.")
        acc = None
    else:
        log_model = LogisticRegression(featuresCol="features", labelCol="label", maxIter=30).fit(df_cls)
        try:
            acc = float(log_model.summary.accuracy)
        except Exception:
            acc = None

    ml_results["classification"] = {
        "target_used": target,
        "median": float(median_val),
        "accuracy": acc,
        "num_classes": distinct_labels,
    }
    print("  Classes:", distinct_labels)
    print("  Accuracy:", acc)

    # KMeans
    print("\n3️⃣ KMeans clustering")
    df_feat = get_features_df()
    df_km = df_feat.select("features").sample(False, 0.10, seed=777)

    km_model = KMeans(k=kmeans_k, seed=42, maxIter=10, initSteps=2).fit(df_km)
    ml_results["kmeans"] = {
        "k": kmeans_k,
        "first_center": km_model.clusterCenters()[0].tolist(),
    }
    print("  k:", kmeans_k)

    # Outlier detection (IQR)
    print("\n4️⃣ Outlier detection (IQR)")
    df_small_read = spark.read.parquet(df_small_path)
    out_cols = use_cols[:3]
    out_counts = []

    for c in out_cols:
        q1, q3 = df_small_read.approxQuantile(c, [0.25, 0.75], 0.05)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        cnt = df_small_read.filter((F.col(c) < lower) | (F.col(c) > upper)).count()
        out_counts.append((c, int(cnt)))
        print(f"  {c}: {cnt} outliers")

    outlier_stats_df = spark.createDataFrame(out_counts, ["column", "outlier_count"])
    display(outlier_stats_df)

    ml_results["outliers"] = {
        "columns": out_cols,
        "counts": [r.asDict() for r in outlier_stats_df.collect()],
    }

    dbutils.fs.put(
        f"{out_base}/ml_results.json",
        json.dumps(ml_results, indent=2),
        overwrite=True,
    )
    outlier_stats_df.write.mode("overwrite").parquet(f"{out_base}/outliers_per_column.parquet")

    print("\n✔️ All ML jobs finished.")

# --------------------------
# 8) Benchmark [1/2/4/8]
# --------------------------
if run_benchmark and run_ml:
    try:
        print("\n---------------------------------------------- Performance Benchmark ----------------------------------------------")

        def speedup_eff(t1, tp, p):
            sp = (t1 / tp) if tp > 0 else None
            eff = (sp / p) if sp is not None else None
            return sp, eff

        parallel_settings = [1, 2, 4, 8]
        REPEATS = 1
        rows = []

        df_bench = spark.read.parquet(df_small_path)
        bench_rows = df_bench.count()
        print(f"Benchmark dataset size: {bench_rows:,} rows")

        def get_bench_features(p):
            df = spark.read.parquet(df_small_path).repartition(p)
            return assembler.transform(df).select(*use_cols, "features")

        def get_bench_small(p):
            return spark.read.parquet(df_small_path).repartition(p)

        def bench_reg(p):
            df_feat = get_bench_features(p)
            d = df_feat.select(F.col(target).alias("label"), "features")
            _ = LinearRegression(maxIter=20).fit(d)

        def bench_cls(p):
            df_feat = get_bench_features(p)
            d = df_feat.withColumn(
                "label",
                F.when(F.col(target) >= median_val, 1).otherwise(0),
            ).select("label", "features")
            _ = LogisticRegression(maxIter=20).fit(d)

        def bench_km(p):
            df_feat = get_bench_features(p)
            d = df_feat.select("features").sample(False, 0.10, seed=777)
            _ = KMeans(k=kmeans_k, seed=42, maxIter=10).fit(d)

        def bench_out(p):
            d = get_bench_small(p).select(*out_cols).dropna()
            for c in out_cols:
                q1, q3 = d.approxQuantile(c, [0.25, 0.75], 0.10)
                iqr = q3 - q1
                lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
                _ = d.filter((F.col(c) < lower) | (F.col(c) > upper)).count()

        bench_jobs = [
            ("regression", bench_reg),
            ("classification", bench_cls),
            ("kmeans", bench_km),
            ("outliers", bench_out),
        ]

        for name, fn in bench_jobs:
            times_map = {}
            for p in parallel_settings:
                try:
                    best = None
                    for rep in range(REPEATS):
                        t0 = time.time()
                        fn(p)
                        t = time.time() - t0
                        best = t if best is None else min(best, t)
                    times_map[p] = best
                except Exception:
                    times_map[p] = None

            if 1 in times_map and times_map[1] is not None:
                t1 = times_map[1]
                for p in parallel_settings:
                    if times_map.get(p) is not None:
                        sp, eff = speedup_eff(t1, times_map[p], p)
                        rows.append((name, p, float(times_map[p]), float(sp), float(eff)))

        if rows:
            bench_df = spark.createDataFrame(
                rows,
                ["job", "parallelism", "time_sec", "speedup_vs_1", "efficiency"],
            )

            bench_df_display = (
                bench_df
                .withColumn("Time (sec)", F.round(F.col("time_sec"), 2))
                .withColumn("Speedup", F.round(F.col("speedup_vs_1"), 2))
                .withColumn("Efficiency (%)", F.round(F.col("efficiency") * 100, 1))
                .drop("time_sec", "speedup_vs_1", "efficiency")
                .orderBy("job", "parallelism")
            )

            display(bench_df_display)

            bench_df.write.mode("overwrite").parquet(f"{out_base}/benchmark_all_results.parquet")
            dbutils.fs.put(
                f"{out_base}/benchmark_all_results.json",
                json.dumps([r.asDict() for r in bench_df.collect()], indent=2),
                overwrite=True,
            )

            p_target = 8
            best_candidates = (
                bench_df.filter(F.col("parallelism") == p_target)
                .orderBy(F.desc("speedup_vs_1"), F.desc("efficiency"))
            )
            best_row = best_candidates.limit(1).collect()
            if best_row:
                best_job = best_row[0]["job"]
                best_speedup = best_row[0]["speedup_vs_1"]
                best_eff = best_row[0]["efficiency"]
                print(f"\n Best scalable job at p=8: {best_job} "
                f"(Speedup = {best_speedup:.2f}x, Efficiency = {best_eff*100:.1f}%)"
                )

        else:
            print("No successful benchmark runs.")

    except Exception as e:
        print("Benchmark failed:", str(e))

print("\nDone.")
print("Results path   :", out_base)
print("Prepared inputs:", input_dir)


In [0]:
from pyspark.sql import functions as F
import plotly.express as px
import plotly.graph_objects as go

P_VALS = [1, 2, 4, 8]
DISCRETE_COLORS = ["#2563EB", "#EF4444", "#10B981", "#A855F7", "#F59E0B", "#14B8A6"]

print("Reading results from:", out_base)

# ----------------------------------
# 1) Load result tables from disk
# ----------------------------------
try:
    bench_df = spark.read.parquet(f"{out_base}/benchmark_all_results.parquet")
    print(f"✓ Benchmark results loaded ({bench_df.count()} rows)")
except Exception as e:
    print("No benchmark results found.")
    print("Reason:", str(e)[:200])
    bench_df = None

try:
    null_stats_df = spark.read.parquet(f"{out_base}/nulls.parquet")
    print(f"✓ Null stats loaded ({null_stats_df.count()} rows)")
except Exception as e:
    print("No null stats found.")
    print("Reason:", str(e)[:200])
    null_stats_df = None

try:
    dtypes_df = spark.read.parquet(f"{out_base}/dtypes.parquet")
    print(f"✓ Data types loaded ({dtypes_df.count()} rows)")
except Exception as e:
    print("No data types file found.")
    print("Reason:", str(e)[:200])
    dtypes_df = None

viz_dir = f"{out_base}/visualizations"
try:
    dbutils.fs.mkdirs(viz_dir)
    print("Visualizations will be saved under:", viz_dir)
except Exception as e:
    print("Could not create visualization directory:", str(e)[:200])

print("\n" + "=" * 60)
print("VISUALIZATIONS")
print("=" * 60)

# ----------------------------------
# 2) Missing values overview
# ----------------------------------
if null_stats_df is not None:
    print("\n[1] Top 10 columns with missing values")

    null_pd = (
        null_stats_df
        .orderBy(F.desc("null_pct"))
        .limit(10)
        .toPandas()
    )

    if len(null_pd) > 0:
        fig1 = px.bar(
            null_pd,
            x="null_pct",
            y="column",
            orientation="h",
            title="Top 10 Columns with Missing Values",
            labels={"null_pct": "Missing %", "column": "Column"},
            color="null_pct",
            color_continuous_scale="Reds",
        )
        fig1.update_layout(
            template="plotly_white",
            showlegend=False,
            height=460,
            margin=dict(l=40, r=30, t=60, b=40),
            xaxis_title="Percentage of Missing Values (%)",
            yaxis_title="Column Name",
            font=dict(size=13),
        )
        display(fig1)

        try:
            fig1.write_html(f"{viz_dir}/viz1_missing_top10.html")
        except Exception as e:
            print("Could not save viz1:", str(e)[:200])

# ----------------------------------
# 3) Data types distribution
# ----------------------------------
if dtypes_df is not None:
    print("\n[2] Data types distribution")

    dtypes_pd = dtypes_df.groupBy("group").count().toPandas()

    if len(dtypes_pd) > 0:
        fig2 = px.pie(
            dtypes_pd,
            values="count",
            names="group",
            title="Distribution of Column Types",
            color_discrete_sequence=["#FF6B6B", "#4ECDC4", "#FFE66D", "#A855F7", "#2563EB"],
        )
        fig2.update_traces(textposition="inside", textinfo="percent+label")
        fig2.update_layout(
            template="plotly_white",
            height=420,
            margin=dict(l=30, r=30, t=60, b=30),
            font=dict(size=13),
        )
        display(fig2)

        try:
            fig2.write_html(f"{viz_dir}/viz2_dtypes_pie.html")
        except Exception as e:
            print("Could not save viz2:", str(e)[:200])

# ----------------------------------
# 4) Benchmark charts
# ----------------------------------
if bench_df is not None:
    bench_pd = bench_df.toPandas()

    # 4.1 Execution time vs workers
    print("\n[3] Execution time vs number of workers")

    fig3 = px.line(
        bench_pd,
        x="parallelism",
        y="time_sec",
        color="job",
        markers=True,
        title="Execution Time vs Number of Workers",
        labels={
            "parallelism": "Workers (Parallelism)",
            "time_sec": "Time (seconds)",
            "job": "Job",
        },
        color_discrete_sequence=DISCRETE_COLORS,
    )
    fig3.update_layout(
        template="plotly_white",
        height=520,
        hovermode="x unified",
        margin=dict(l=45, r=30, t=70, b=45),
        font=dict(size=13),
    )
    fig3.update_xaxes(tickmode="array", tickvals=P_VALS)
    display(fig3)

    try:
        fig3.write_html(f"{viz_dir}/viz3_exec_time.html")
    except Exception as e:
        print("Could not save viz3:", str(e)[:200])

    # 4.2 Speedup vs workers
    print("\n[4] Speedup vs number of workers")

    fig4 = go.Figure()
    for job in bench_pd["job"].unique():
        job_data = bench_pd[bench_pd["job"] == job].sort_values("parallelism")
        fig4.add_trace(
            go.Scatter(
                x=job_data["parallelism"],
                y=job_data["speedup_vs_1"],
                mode="lines+markers",
                name=str(job),
                marker=dict(size=10),
                line=dict(width=3),
            )
        )

    fig4.add_trace(
        go.Scatter(
            x=P_VALS,
            y=P_VALS,
            mode="lines",
            name="Ideal linear speedup",
            line=dict(width=2, dash="dash", color="#6B7280"),
        )
    )

    fig4.update_layout(
        title="Speedup vs Number of Workers",
        xaxis_title="Workers (Parallelism)",
        yaxis_title="Speedup (relative to p=1)",
        template="plotly_white",
        height=520,
        hovermode="x unified",
        margin=dict(l=45, r=30, t=70, b=45),
        font=dict(size=13),
        legend_title="Job",
    )
    fig4.update_xaxes(tickmode="array", tickvals=P_VALS)
    display(fig4)

    try:
        fig4.write_html(f"{viz_dir}/viz4_speedup.html")
    except Exception as e:
        print("Could not save viz4:", str(e)[:200])

    # 4.3 Efficiency vs workers
    print("\n[5] Efficiency vs number of workers")

    bench_pd["efficiency_pct"] = bench_pd["efficiency"] * 100

    fig5 = px.bar(
        bench_pd,
        x="parallelism",
        y="efficiency_pct",
        color="job",
        barmode="group",
        title="Efficiency vs Number of Workers",
        labels={
            "parallelism": "Workers (Parallelism)",
            "efficiency_pct": "Efficiency (%)",
            "job": "Job",
        },
        color_discrete_sequence=DISCRETE_COLORS,
    )
    fig5.add_hline(
        y=100,
        line_dash="dash",
        line_color="#6B7280",
        annotation_text="100% (ideal)",
        annotation_position="top right",
    )
    fig5.update_layout(
        template="plotly_white",
        height=520,
        hovermode="x unified",
        margin=dict(l=45, r=30, t=70, b=45),
        font=dict(size=13),
    )
    fig5.update_xaxes(tickmode="array", tickvals=P_VALS)
    display(fig5)

    try:
        fig5.write_html(f"{viz_dir}/viz5_efficiency.html")
    except Exception as e:
        print("Could not save viz5:", str(e)[:200])

    # 4.4 Small text summary for p=8

    print("SUMMARY (p = 8 workers)")
  

    p8 = bench_pd[bench_pd["parallelism"] == 8].sort_values("speedup_vs_1", ascending=False)
    if len(p8) > 0:
        best = p8.iloc[0]
        print(f"\nBest job at 8 workers: {str(best['job']).upper()}")
        print(f"  Speedup : {best['speedup_vs_1']:.2f}x")
        print(f"  Efficiency: {best['efficiency'] * 100:.1f}%")
        print(f"  Time    : {best['time_sec']:.2f}s")

print("\nAll visualizations finished.")

